In [3]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever

# Chunk parameters
chunk_size = 256
chunk_overlap = 32

docs = CSVLoader(
    file_path='../../data/raw/hyperbook.csv', 
    content_columns=['content'], 
    metadata_columns=['url', 'title', 'authors'],
    encoding='utf-8'
).load()

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
docs = text_splitter.split_documents(docs)

sparse_retriever = BM25Retriever.from_documents(docs)

In [10]:
docs = sparse_retriever.invoke('tardigrades')
docs[0].metadata

{'source': '../../data/raw/hyperbook.csv',
 'row': 22,
 'url': 'https://bioengineering.hyperbook.mcgill.ca/the-physical-adaptations-underlying-the-tardigrades-resistance-to-extreme-environmental-conditions/',
 'title': 'The Physical Adaptations Underlying the Tardigrade’s Resistance to Extreme Environmental Conditions\xa0',
 'authors': 'Luccia Jabbour, Eden Karp-Foster, Jeremy Lachance, Wadi Zahka'}

In [ ]:
SOURCE_FORMAT = '{title}\n{authors}\n{url}'

def get_sources(docs) -> str:
    """
    Returns a string with all the sources
    formatted by title, authors and url.
    """
    sources = set() # Keep unique sources

    for doc in docs:
        metadata = doc.metadata
        source = SOURCE_FORMAT.format(
            title=metadata['title'],
            authors=metadata['authors'],
            url=metadata['url']
        )
        sources.add(source)

    # Concatenate into a string
    sources = '\n\n'.join(sources)

    return sources

In [63]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool

@tool('retrieval_tool', parse_docstring=True)
def multiplication_tool(int_a, int_b):
    """
    Calculates the multiplication of int_a and int_b
    
    args:
        int_a: first integer
        int_b: second integer
    """

    return int_a*int_b

model = AzureChatOpenAI(azure_deployment='gpt-4o', api_version='2024-10-21')
response = model.bind_tools([multiplication_tool]).invoke(['What is 4 times 4? Use a tool'])

In [64]:
response

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_gslcxmTEe0wRBJ6ER32A4F9W', 'function': {'arguments': '{"int_a":4,"int_b":4}', 'name': 'retrieval_tool'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 78, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-11-20', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-BrmKTfPJJe85kN7BgYVipzkaJfsIj', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}]

In [65]:
response.tool_calls

[{'name': 'retrieval_tool',
  'args': {'int_a': 4, 'int_b': 4},
  'id': 'call_gslcxmTEe0wRBJ6ER32A4F9W',
  'type': 'tool_call'}]